In [1]:
import pandas as pd
import numpy as np
import ast
import re
from collections import Counter
import json

import load_data as ld
import candidates as ds
import priority_queue as pq
import subgroups as s
import refinements as rf
import subgroup_selection as ss
import search as bs
import preprocessing as pp

In [2]:
df = pd.read_csv('algebrakit_firstbatch_2.csv')

# Set display option to show all columns
pd.set_option('display.max_columns', None)

# Show the columns 
display(df.head(1))

,exerciseID,sessionID,sessionSuccess,creationTimestamp,finished,marksEarned,marksTotal,skillName,stepType,errors,questionID,questionType,itemType,itemContent,interactionID,interactionType,interactionSolution,interactionDerivation,result.status,result.progress,result.events,result.tags,result.scoring.marksEarned,result.scoring.marksTotal,result.scoring.penalties.marksPenalty,result.scoring.penalties.hintsRequested,result.scoring.penalties.mathErrors,result.scoring.finished,result.scoring.maxAttempts,result.scoring.usedAttempts,result.scoring.failed
0,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,False,1.0,4.0,['FractionArithmeticMultiplicationProper'],['CALCULATE'],[nan],Q1,QUESTION,TEXT,$$n = 4\cdot c \cdot c \cdot c $$,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
data_path = 'Data/private_data/sessionInfo' 
exID = '0beaa764-f558-4124-8d4d-b224ebdb2d4a'
seID = '0c38bfc2-c44e-4574-a255-29662f9fb198'

In [4]:
data_dict = pp.open_library(data_path=data_path)


df = pp.exercise_dataframe(data_dict)
df.head()

,exerciseID,sessionID,success,creationTimestamp,skills,stepTypes,finished,marksEarned,marksTotal,questionID,type,items,interactionID,itemType,interactionType,result,solution,derivation,content
0,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,[FractionArithmeticMultiplicationProper],[CALCULATE],False,1.0,4.0,Q1,QUESTION,"{'itemType': 'TEXT', 'content': '$$n = 4\cdot ...",Ex8wE,INTERACTION,MULTISTEP,"{'status': 'FINISHED', 'progress': 1, 'events'...",$$n =4 c ^{3}$$,[{'expression': '$$n =4 c \times c \times c $$...,NaN
1,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,[FractionArithmeticMultiplicationProper],[CALCULATE],False,1.0,4.0,Q1,QUESTION,"{'id': 'Ex8wE', 'itemType': 'INTERACTION', 'in...",Ex8wE,INTERACTION,MULTISTEP,"{'status': 'FINISHED', 'progress': 1, 'events'...",$$n =4 c ^{3}$$,[{'expression': '$$n =4 c \times c \times c $$...,NaN
2,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,[FractionArithmeticMultiplicationProper],[CALCULATE],False,1.0,4.0,Q2,QUESTION,"{'itemType': 'TEXT', 'content': '$$u = t \cdot...",gBslt,INTERACTION,MULTISTEP,"{'status': 'GIVEUP', 'progress': 0, 'events': ...",$$u =\frac{4}{25} t ^{3}$$,[{'expression': '$$u =\frac{2}{5}\times \frac{...,NaN
3,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,[FractionArithmeticMultiplicationProper],[CALCULATE],False,1.0,4.0,Q2,QUESTION,"{'id': 'gBslt', 'itemType': 'INTERACTION', 'in...",gBslt,INTERACTION,MULTISTEP,"{'status': 'GIVEUP', 'progress': 0, 'events': ...",$$u =\frac{4}{25} t ^{3}$$,[{'expression': '$$u =\frac{2}{5}\times \frac{...,NaN
4,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,[FractionArithmeticMultiplicationProper],[CALCULATE],False,1.0,4.0,Q3,QUESTION,"{'itemType': 'TEXT', 'content': '$$p = v \cdot...",OPqyZ,INTERACTION,MULTISTEP,"{'status': 'VIRGIN', 'progress': 0, 'events': ...",$$p =48 v ^{3}$$,[{'expression': '$$p =16\times 3 v \times v \t...,NaN


In [5]:
df['items']

0      {'itemType': 'TEXT', 'content': '$$n = 4\cdot ...
1      {'id': 'Ex8wE', 'itemType': 'INTERACTION', 'in...
2      {'itemType': 'TEXT', 'content': '$$u = t \cdot...
3      {'id': 'gBslt', 'itemType': 'INTERACTION', 'in...
4      {'itemType': 'TEXT', 'content': '$$p = v \cdot...
                             ...                        
875    {'id': 'aEDsA', 'itemType': 'INTERACTION', 'in...
876    {'itemType': 'TEXT', 'content': '$$y = 3x - 10...
877    {'id': 'HuaWk', 'itemType': 'INTERACTION', 'in...
878    {'itemType': 'TEXT', 'content': '$$y = - x + 6...
879    {'id': 'lwOZO', 'itemType': 'INTERACTION', 'in...
Name: items, Length: 10689, dtype: object